## Tugas Mandiri Pemrograman Paralel Pertemuan 4
**Nama: Prima Azahra     NIM: 2411511005**  

### Tugas 1 - Bank Account Race Condition

In [1]:
import threading
import time

class BankAccount:
    def __init__(self, initial_balance=1000000):
        self.balance = initial_balance
        self.lock = threading.Lock()

    # Operasi tanpa lock 
    def withdraw_deposit_unlocked(self):
        # Operasi "tarik 1 lalu setor 1" dilakukan 10.000 kali
        for _ in range(10000):
            self.balance -= 1
            self.balance += 1

    # Operasi dengan menggunakan threading.Lock()
    def withdraw_deposit_locked(self):
        for _ in range(10000):
            with self.lock:
                self.balance -= 1
                self.balance += 1

# Simulasi: tanpa lock
print("Simulasi Tanpa Lock")
account_no_lock = BankAccount(1000000)
threads_no_lock = []

for i in range(10):
    t = threading.Thread(target=account_no_lock.withdraw_deposit_unlocked)
    threads_no_lock.append(t)
    t.start()

for t in threads_no_lock:
    t.join()

print(f"Saldo akhir tanpa lock: {account_no_lock.balance}")

Simulasi Tanpa Lock
Saldo akhir tanpa lock: 1000000


In [2]:
# Simulasi 2: threading.lock()
print("\nSimulasi Dengan Threading.Lock()")
account_with_lock = BankAccount(1000000)
threads_with_lock = []

for i in range(10):
    t = threading.Thread(target=account_with_lock.withdraw_deposit_locked)
    threads_with_lock.append(t)
    t.start()

for t in threads_with_lock:
    t.join()

print(f"Saldo akhir dengan lock: {account_with_lock.balance}")


Simulasi Dengan Threading.Lock()
Saldo akhir dengan lock: 1000000


**Penjelasan**                      
Pada latihan nomor satu, kita diminta buat simulasi rekening bank dengan saldo awal satu juta rupiah, lalu bikin sepuluh thread yang masing-masing menjalankan operasi tarik satu dan setor satu sebanyak sepuluh ribu kali. Dari hasil percobaan, kalau kita jalankan programnya tanpa pakai lock, saldo akhir sering kali berubah-ubah alias tidak pas satu juta lagi. Hal ini terjadi karena ada masalah race condition, di mana beberapa thread mengakses dan mengubah variabel saldo secara bersamaan sehingga terjadi tumpang tindih proses (lost update). Untuk mengatasi masalah tersebut, kita perlu menambahkan threading.Lock() di bagian kodenya. Dengan memakai lock, operasi pengurangan dan penambahan saldo menjadi bersifat eksklusif, artinya thread lain harus antri terlebih dulu sampai thread yang sedang berjalan selesai. Hasilnya, saldo akhir bisa konsisten kembali ke angka satu juta rupiah dengan aman.

### Tugas 2 - Poducer-Consumer dengan Banyak Consumer

In [3]:
import threading
import queue
import time
import random

def producer(q, jumlah_item, jumlah_consumer):
    for i in range(1, jumlah_item + 1):
        item = f"item-{i}"
        time.sleep(random.uniform(0.05, 0.2))  # simulasi waktu produksi
        q.put(item)
        print(f"[Producer] menghasilkan: {item}")
    
    # Kirim sinyal None sebanyak jumlah consumer agar semuanya berhenti dengan benar
    for _ in range(jumlah_consumer):
        q.put(None)
    print("[Producer] Selesai menghasilkan semua item dan mengirim sinyal berhenti.")

def consumer(q, nama):
    while True:
        item = q.get()
        if item is None:
            q.task_done()
            break
        time.sleep(random.uniform(0.05, 0.15))  # simulasi waktu pemrosesan
        print(f"      [{nama}] memproses: {item}")
        q.task_done()
    print(f"[{nama}] Berhenti.")

# Inisialisasi Queue dan parameter
q = queue.Queue()
jumlah_item = 9
jumlah_consumer = 3

# Membuat thread Producer (1 buah)
t_producer = threading.Thread(target=producer, args=(q, jumlah_item, jumlah_consumer))

# Membuat thread Consumer (3 buah menggunakan loop)
t_consumers = []
for i in range(1, jumlah_consumer + 1):
    t = threading.Thread(target=consumer, args=(q, f"Consumer-{i}"))
    t_consumers.append(t)

# Menjalankan semua thread
t_producer.start()
for t in t_consumers:
    t.start()

# Menunggu thread producer selesai
t_producer.join()

# Menunggu semua consumer selesai memproses sisa antrean
q.join()

# Menunggu seluruh thread consumer bergabung kembali
for t in t_consumers:
    t.join()

print("\nSemua item sudah diproduksi dan dikonsumsi oleh 3 consumer.")

[Producer] menghasilkan: item-1
      [Consumer-1] memproses: item-1
[Producer] menghasilkan: item-2
      [Consumer-2] memproses: item-2
[Producer] menghasilkan: item-3
      [Consumer-3] memproses: item-3
[Producer] menghasilkan: item-4
[Producer] menghasilkan: item-5
      [Consumer-1] memproses: item-4
[Producer] menghasilkan: item-6
      [Consumer-2] memproses: item-5
      [Consumer-3] memproses: item-6
[Producer] menghasilkan: item-7
      [Consumer-1] memproses: item-7
[Producer] menghasilkan: item-8
      [Consumer-2] memproses: item-8
[Producer] menghasilkan: item-9
[Producer] Selesai menghasilkan semua item dan mengirim sinyal berhenti.
[Consumer-1] Berhenti.
[Consumer-2] Berhenti.
      [Consumer-3] memproses: item-9
[Consumer-3] Berhenti.

Semua item sudah diproduksi dan dikonsumsi oleh 3 consumer.


**Penjelasan**     
Producer di kode ini bertugas membuat data dan memasukkannya ke queue, lalu mengirim sinyal None sebanyak 3 kali di akhir supaya kebagian semua oleh 3 consumer. Di sisi lain, ketiga fungsi consumer bakal terus mengambil data dari queue sampai ketemu sinyal None tersebut, yang kemudian dipakai buat nge-break loop dan menghentikan thread secara bersih.

### Tugas 3 - Eksperimen ThreadPoolExecutor

In [4]:
import concurrent.futures
import time

# Simulasi fungsi unduh/proses file
def simulasi_unduh(file_id):
    # Simulasi waktu pengerjaan suatu tugas (misal: 0.1 detik)
    time.sleep(0.1)
    return f"File-{file_id} selesai"

# Daftar jumlah file yang akan diproses
total_file = 50
list_max_workers = [2, 4, 8, 16]

for mw in list_max_workers:
    start_time = time.time()
    
    # Menggunakan ThreadPoolExecutor dengan max_workers berbeda-beda
    with concurrent.futures.ThreadPoolExecutor(max_workers=mw) as executor:
        futures = [executor.submit(simulasi_unduh, i) for i in range(1, total_file + 1)]
        concurrent.futures.wait(futures)
        
    end_time = time.time()
    durasi = end_time - start_time
    print(f"max_workers = {mw:2d} | Waktu Eksekusi: {durasi:.4f} detik")

max_workers =  2 | Waktu Eksekusi: 2.5307 detik
max_workers =  4 | Waktu Eksekusi: 1.3093 detik
max_workers =  8 | Waktu Eksekusi: 0.7055 detik
max_workers = 16 | Waktu Eksekusi: 0.4053 detik


**Penjelasan**            
Kode di atas menggunakan ThreadPoolExecutor untuk memproses 50 tugas secara paralel dengan membandingkan parameter max_workers bernilai 2, 4, 8, dan 16.          
Kenapa menambah max_workers terus-menerus tidak selalu mempercepat proses? Karena ada titik jenuh (bottleneck). Saat jumlah worker sudah melampaui batas optimal (atau batasan sumber daya CPU/I/O dan overhead manajemen thread di Python seperti GIL), penambahan thread justru membuat sistem menghabiskan waktu untuk bergantian (context switching) alih-alih mempercepat eksekusi.

### Tugas 4 - Refleksi Singkat
**Contoh kasus nyata di mana threading Python membantu:** Proses web scraping atau pengunduhan banyak gambar/dokumen secara bersamaan dari internet. Threading sangat membantu di sini karena program tidak perlu menunggu satu unduhan selesai untuk mulai mengunduh file berikutnya, sehingga waktu total jauh lebih efisien.

**Contoh kasus di mana threading tidak membantu dan sebaiknya menunggu multiprocessing:** Proses pengolahan citra (image processing) beresolusi tinggi, enkripsi data yang berat, atau perhitungan matematis yang masif secara paralel. Threading tidak akan mempercepat proses ini karena adanya GIL (Global Interpreter Lock) di Python yang membuat eksekusi kode terikat pada satu inti CPU saja, sehingga tugas-tugas yang memakan kapasitas CPU (CPU-bound) harus diselesaikan menggunakan multiprocessing.